In [1]:
import timm
model = timm.create_model(
    "vit_large_patch16_dinov3.lvd1689m",
    pretrained=True,
    num_classes=0,   # без головы классификатора
).to("cuda").bfloat16()

C:\Users\Dima\PycharmProjects\lct-2026\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Краткая сводка
print(model)                      # дерево модулей
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")

# Все имена параметров
for name, p in model.named_parameters():
    print(name, tuple(p.shape))

# Доступ к конкретному блоку
blk = model.blocks[0]
print(blk.attn.qkv.weight.shape)   # [3*D, D]
print(blk.mlp.fc1.weight.shape)

# Веса как numpy
w = model.blocks[0].attn.qkv.weight.detach().cpu().numpy()

Eva(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (rope): RotaryEmbeddingDinoV3()
  (norm_pre): Identity()
  (blocks): ModuleList(
    (0-23): 24 x EvaBlock(
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): EvaAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path1): Identity()
      (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, 

In [1]:
images_dir = "D:/Downloads/dataset/images"

In [4]:
import torch
import cv2

image = cv2.imread(images_dir, cv2.IMREAD_COLOR)
image_tensor = torch.tensor(image, dtype=torch.float32)

In [5]:
image_tensor.unsqueeze(0).transpose(1,3).transpose(2,3).shape

torch.Size([1, 3, 1080, 1920])

In [6]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
transform =  A.Compose( [A.Resize(960,272),  A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
image_tr = transform(image=image)

In [1]:
result = model(image_tr["image"].unsqueeze(0).to("cuda").bfloat16())

NameError: name 'model' is not defined

In [8]:
result


tensor([[-0.0762, -0.1660, -0.0161,  ...,  0.0942, -0.0151, -0.0356]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward1>)

In [2]:
import pandas as pd

df = pd.read_csv("D:/Downloads/dataset/train.csv")

In [3]:
from PIL import Image
import numpy as np
from src.vehicle_reid.utils.bbox import BBox
%matplotlib inline
import cv2
from matplotlib import pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
for im in df[df["vehicle_id"] == 0].iterrows():
    print(images_dir +"/"+ im[1].image_id+ ".jpg")
    image = cv2.imread(images_dir +"/" + im[1].image_id+ ".jpg")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    bbox = BBox(x=float(im[1].x), y=float(im[1].y), w=float(im[1].w), h=float(im[1].h))
    x1, y1, x2, y2 =  bbox.to_xyxy(im_y=image.shape[0],im_x=image.shape[1],patch_size=16)
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    print(x1, y1, x2, y2)
    print(bbox.x, bbox.y, bbox.w, bbox.h)
    print(image.shape)
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)
    crop = A.Compose([A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])(image=image[y1:y2, x1:x2])["image"]
    print( crop.shape)
    #plt.imshow(image[y1:y2, x1:x2])
    #plt.axis('off')
    #plt.show()

C:\Users\Dima\PycharmProjects\lct-2026\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:/Downloads/dataset/images/5066e220c9e847008a65030d41547512.jpg
815 1 1343 577
815.0 1.0 526.0 568.0
(1080, 1920, 3)
torch.Size([3, 576, 528])
D:/Downloads/dataset/images/cd767cf1d4d842aca62f385ae4c272ba.jpg
1275 156 1915 604
1275.0 156.0 644.0 432.0
(1080, 1920, 3)
torch.Size([3, 448, 640])
D:/Downloads/dataset/images/bb84b3ebd6094a67895b9f73a4606114.jpg
838 126 1478 910
838.0 126.0 627.0 774.0
(1080, 1920, 3)
torch.Size([3, 784, 640])
D:/Downloads/dataset/images/34b254d8cb3d47c897f8f7bcb4eda835.jpg
192 452 1136 964
192.0 452.0 931.0 498.0
(1080, 1920, 3)
torch.Size([3, 512, 944])
D:/Downloads/dataset/images/d594bbcd765448e189e21a750ad00097.jpg
831 127 1471 879
831.0 127.0 631.0 736.0
(1080, 1920, 3)
torch.Size([3, 752, 640])
D:/Downloads/dataset/images/1313d83b6c8d405694051b82854f9678.jpg
716 11 1260 619
716.0 11.0 530.0 600.0
(1080, 1920, 3)
torch.Size([3, 608, 544])
D:/Downloads/dataset/images/8181405f7bfd4874a4e451a1fd7e0801.jpg
193 452 1137 964
193.0 452.0 928.0 496.0
(1080, 192

In [22]:
df.groupby("vehicle_id").size().count()

np.int64(1541)

In [68]:
 (65) + 16 - (65) % 16


80